# OpenDataZurich starter notebook

This notebook runs **entirely in your browser** (JupyterLite with a Pyodide Python kernel): nothing is installed and your changes are stored only in this browser. Run the cells from top to bottom with <kbd>Shift</kbd>+<kbd>Enter</kbd>, or use *Run → Run All Cells*.

In [ ]:
# Install the OpenDataZurich helper and missingno into the in-browser Python
import sys

import pyarrow  # load before pandas, so pandas can use it (parquet files, fast strings)
import pandas as pd
import piplite

await piplite.install(["opendata-lite", "missingno"])
print(f"Python {sys.version.split()[0]}, pandas {pd.__version__} (Pyodide, in your browser)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from opendata_lite import OpenDataZurich

odz = OpenDataZurich()

In [ ]:
package_id = "geo__gdp__inventar_der_schuetzenswerten_gaerten_und_anlagen_von_kommunaler_bedeutung_der_stadt_"
resource_id = None

In [ ]:
# Use the name of a dataset, e.g., from the OpenDataZurich site or the list above, to get the package
package = await odz.get_package(name=package_id)

In [ ]:
package.display_metadata()

In [ ]:
package.display_resource_summary()

In [ ]:
package.geo_resource_metadata_df.head()[["format"]]

In [ ]:
resource = package.geo_resource(id=resource_id) if resource_id is not None and resource_id.upper() != "NONE" else package.geo_resource(0)
resource.display_metadata()

In [ ]:
df = await resource.load()

In [ ]:
print("The resource includes the following geo layers:", 
      await resource.get_layers(),
      """By default, the first layer is used in this notebook. To use a different layer, uncomment the code snippet in this cell and replace "insert-layer-name-here" with the name of your desired layer.""",
     sep="\n")
# df = await resource.layer_df("insert-layer-name-here")

In [ ]:
df.plot()

In [ ]:
# get background map with "Stadtkreise"
stadtkreise_package = await odz.get_package(name="geo_stadtkreise")
stadtkreise_resource = stadtkreise_package.geo_resource(0)
background_df = await stadtkreise_resource.layer_df("adm_stadtkreise_a")

In [ ]:
ax = background_df.plot(color="tab:gray", alpha=0.2)
background_df.boundary.plot(ax=ax, color="tab:blue", linewidth=0.5, alpha=0.7)
df.plot(ax=ax, color="tab:red", alpha=0.3, edgecolors='none')

In [ ]:
# drop columns that have no values
df.dropna(how='all', axis=1, inplace=True)

In [ ]:
print(f'The dataset has {df.shape[0]:,.0f} rows (observations) and {df.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {df.duplicated().sum()} exact duplicates in the data.')

In [ ]:
df.info(memory_usage='deep', verbose=True)

In [ ]:
df.head()

In [ ]:
# display a small random sample transposed in order to see all variables
df.sample(3).T

In [ ]:
# describe non-numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(exclude='number'))
except:
    print("No categorical data in dataset.")

In [ ]:
# describe numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(include='number'))
except:
    print("No numercial data in dataset.")

In [ ]:
# check missing values with missingno
# https://github.com/ResidentMario/missingno
import missingno as msno
msno.matrix(df, labels=True, sort='descending');

In [ ]:
# plot a histogram for each numerical feature
try:
    df.hist(bins=25, rwidth=.9)
    plt.tight_layout()
    plt.show()
except:
    print("No numercial data to plot.")

## Need more power, or R?

This in-browser kernel is limited to the memory of your browser tab (a few GB at most) and cannot install packages with compiled code that are not part of Pyodide. For large datasets, other Python packages or for working in R, open the same dataset in a full Renku session:

* Python: <https://renkulab.io/p/opendatazurich/starter-code>
* R: <https://github.com/opendatazurich/opendatazurich_renku_r>

Use *File → Download* to keep a copy of this notebook.